# table_values_verification

**Source:** `06_analysis/table_values_verification.py`  
**Purpose:** Databricks notebook auto-generated from framework Python module.


## Section 1: Additional module code and configuration

This cell handles: *Additional module code and configuration*


In [ ]:
"""Verify control metadata and silver table availability after orchestration."""

from __future__ import annotations

import json
from typing import Any


## Section 2: Define `_as_rows()` helper function

This cell handles: *Define `_as_rows()` helper function*


In [ ]:
def _as_rows(df):
    return [r.asDict(recursive=True) for r in df.collect()]


## Section 3: Define `_parse_csv_list()` helper function

This cell handles: *Define `_parse_csv_list()` helper function*


In [ ]:
def _parse_csv_list(value: str) -> list[str]:
    return [v.strip() for v in value.split(",") if v.strip()]


## Section 4: Define `_sql_in()` helper function

This cell handles: *Define `_sql_in()` helper function*


In [ ]:
def _sql_in(values: list[str]) -> str:
    escaped = [v.replace("'", "''") for v in values]
    return ", ".join(f"'{v}'" for v in escaped)


## Section 5: Define `run_table_values_verification()` function with logic for processing

This cell handles: *Define `run_table_values_verification()` function with logic for processing*


In [ ]:
def run_table_values_verification(
    spark,
    catalog: str,
    control_schema: str,
    silver_schema: str,
    product_names: list[str],
    source_systems: list[str],
) -> dict:
    result: dict[str, Any] = {}

    print("\n" + "=" * 80)
    print("TABLE DATA CHECK - INGESTION FRAMEWORK")
    print("=" * 80)

    print("\n1. SOURCE REGISTRY - ACTIVE SOURCES")
    print("-" * 80)
    filters = ["lower(cast(is_active as string)) IN ('true', '1', 'yes', 'y', 't')"]
    if product_names:
        filters.append(f"lower(product_name) IN ({_sql_in([p.lower() for p in product_names])})")
    if source_systems:
        filters.append(f"lower(source_system) IN ({_sql_in([s.lower() for s in source_systems])})")

    active_sources_query = f"""
        SELECT
            product_name,
            source_system,
            source_entity,
            source_type,
            is_active,
            landing_table,
            silver_table
        FROM {catalog}.{control_schema}.source_registry
        WHERE {' AND '.join(filters)}
        ORDER BY product_name, source_system, source_entity
    """

    active_sources_df = spark.sql(
        active_sources_query
    )
    active_sources_df.show(50, False)
    result["active_sources"] = _as_rows(active_sources_df)

    print("\n2. CONTROL TABLE ROW COUNTS")
    print("-" * 80)
    counts = {}
    for table in ["source_registry", "column_mapping", "dq_rules", "publish_rules"]:
        cnt = spark.sql(f"SELECT COUNT(*) AS cnt FROM {catalog}.{control_schema}.{table}").collect()[0]["cnt"]
        counts[table] = int(cnt)
        print(f"  {table}: {cnt} rows")
    result["control_counts"] = counts

    print("\n3. ALL SCHEMAS IN CATALOG")
    print("-" * 80)
    schemas_df = spark.sql(f"SHOW SCHEMAS IN {catalog}")
    schemas_df.show(100, False)
    result["schemas"] = _as_rows(schemas_df)

    print("\n4. SILVER TABLES")
    print("-" * 80)
    try:
        silver_df = spark.sql(f"SHOW TABLES IN {catalog}.{silver_schema}")
        silver_df.show(200, False)
        result["silver_tables"] = [r["tableName"] for r in silver_df.collect()]
    except Exception as exc:
        result["silver_tables"] = [f"ERROR: {str(exc)}"]

    print("\n5. SOURCE CONFIG FOR FILTERED FLOWS")
    print("-" * 80)
    source_cfg_df = spark.sql(active_sources_query)
    source_cfg_df.show(20, False)
    result["source_config"] = _as_rows(source_cfg_df)

    print("\n6. PUBLISH RULES FOR FILTERED FLOWS")
    print("-" * 80)
    entities = [r["source_entity"] for r in active_sources_df.select("source_entity").distinct().collect()]
    if entities:
        publish_df = spark.sql(
            f"SELECT * FROM {catalog}.{control_schema}.publish_rules WHERE source_entity IN ({_sql_in(entities)})"
        )
        publish_df.show(200, False)
        result["publish_rules"] = _as_rows(publish_df)
    else:
        result["publish_rules"] = []

    print("\n7. LANDING/SILVER ROW COUNTS (PER ACTIVE SOURCE)")
    print("-" * 80)
    table_checks: list[dict[str, Any]] = []
    for src in active_sources_df.collect():
        rec: dict[str, Any] = {
            "product_name": src["product_name"],
            "source_system": src["source_system"],
            "source_entity": src["source_entity"],
            "landing_table": src["landing_table"],
            "silver_table": src["silver_table"],
            "landing_rows": None,
            "silver_rows": None,
            "landing_error": None,
            "silver_error": None,
        }

        if src["landing_table"]:
            try:
                rec["landing_rows"] = int(
                    spark.sql(f"SELECT COUNT(*) AS c FROM {src['landing_table']}").collect()[0]["c"]
                )
            except Exception as exc:
                rec["landing_error"] = str(exc)

        if src["silver_table"]:
            try:
                rec["silver_rows"] = int(
                    spark.sql(f"SELECT COUNT(*) AS c FROM {src['silver_table']}").collect()[0]["c"]
                )
            except Exception as exc:
                rec["silver_error"] = str(exc)

        table_checks.append(rec)

    checks_df = spark.createDataFrame(table_checks) if table_checks else spark.sql("SELECT 1 WHERE 1=0")
    checks_df.show(200, False)
    result["table_checks"] = table_checks

    print("\n" + "=" * 80)
    print("END OF TABLE DATA CHECK")
    print("=" * 80)
    return result


_dbutils: Any = globals().get("dbutils")
_spark: Any = globals().get("spark")

if _dbutils is None or _spark is None:
    raise RuntimeError("This notebook must run in Databricks where both spark and dbutils are available")

_dbutils.widgets.text("catalog", "eng511_development_bronze")
_dbutils.widgets.text("control_schema", "control_dev")
_dbutils.widgets.text("silver_schema", "silver_dev")
_dbutils.widgets.text("product_names", "pia,connect")
_dbutils.widgets.text("source_systems", "")

catalog = _dbutils.widgets.get("catalog").strip()
control_schema = _dbutils.widgets.get("control_schema").strip()
silver_schema = _dbutils.widgets.get("silver_schema").strip()
product_names = _parse_csv_list(_dbutils.widgets.get("product_names"))
source_systems = _parse_csv_list(_dbutils.widgets.get("source_systems"))

summary = run_table_values_verification(
    _spark,
    catalog=catalog,
    control_schema=control_schema,
    silver_schema=silver_schema,
    product_names=product_names,
    source_systems=source_systems,
)

_dbutils.notebook.exit(json.dumps(summary, indent=2, default=str))
